In [2]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn shap


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.

In [11]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Suppress annoying FutureWarnings for a clean, professional presentation
warnings.simplefilter(action='ignore', category=FutureWarning)

# 2. Set a premium, cohesive aesthetic for all future plots globally
sns.set_theme(
    style="whitegrid", 
    palette="mako", 
    rc={"axes.spines.right": False, "axes.spines.top": False, "figure.figsize": (10, 6)}
)

os.makedirs('../output', exist_ok=True)

def load_data(sentiment_path: str, trader_path: str) -> tuple:
    print("📥 Loading datasets into memory...")
    df_sent = pd.read_csv(sentiment_path)
    df_trad = pd.read_csv(trader_path)
    return df_sent, df_trad

df_sentiment, df_trader = load_data('../data/sentiment.csv', '../data/trader_data.csv')

# Professional output reporting
print("-" * 50)
print(f"📊 Dataset Overview:")
print(f"   ➤ Sentiment Data : {df_sentiment.shape[0]:,} rows, {df_sentiment.shape[1]} columns")
print(f"   ➤ Trader Data    : {df_trader.shape[0]:,} rows, {df_trader.shape[1]} columns")
print("-" * 50)

# Display a beautifully styled preview of the sentiment dataframe
display(
    df_sentiment.head(3).style
    .set_caption("Preview: Bitcoin Market Sentiment")
    .set_table_styles([{
        'selector': 'caption',
        'props': [('color', '#2c3e50'), ('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]
    }, {
        'selector': 'th',
        'props': [('background-color', '#34495e'), ('color', 'white'), ('font-weight', 'bold')]
    }])
    .set_properties(**{'background-color': '#f8f9fa', 'color': '#212529', 'border': '1px solid #dee2e6'})
)


📥 Loading datasets into memory...
--------------------------------------------------
📊 Dataset Overview:
   ➤ Sentiment Data : 2,644 rows, 4 columns
   ➤ Trader Data    : 211,224 rows, 16 columns
--------------------------------------------------


,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03


In [12]:
def clean_and_align_sentiment(df_sentiment: pd.DataFrame) -> pd.DataFrame:
    df = df_sentiment.copy()
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
    # Sort and drop any overlapping dates to ensure clean time-series merge later
    df = df.sort_values('date').reset_index(drop=True).drop_duplicates(subset=['date'])
    df = df[['date', 'value', 'classification']]
    df.rename(columns={'value': 'sentiment_score', 'classification': 'sentiment_class'}, inplace=True)
    return df

def clean_trader_data(df_trader: pd.DataFrame) -> pd.DataFrame:
    df = df_trader.copy()
    # Convert epoch to datetime and floor to daily for merging
    df['datetime_utc'] = pd.to_datetime(df['Timestamp'], unit='ms')
    df['date'] = df['datetime_utc'].dt.floor('d')
    df = df.drop_duplicates()
    return df

print("🧹 Cleaning and aligning datasets...")
df_sentiment_clean = clean_and_align_sentiment(df_sentiment)
df_trader_clean = clean_trader_data(df_trader)
print("✅ Data cleaning complete. Previewing cleaned sentiment data:")

# Maintain consistent, premium styling
display(
    df_sentiment_clean.head(3).style
    .set_caption("Cleaned Data: Sentiment Scores")
    .set_table_styles([{
        'selector': 'caption',
        'props': [('color', '#2c3e50'), ('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]
    }, {
        'selector': 'th',
        'props': [('background-color', '#34495e'), ('color', 'white'), ('font-weight', 'bold')]
    }])
    .set_properties(**{'background-color': '#f8f9fa', 'color': '#212529', 'border': '1px solid #dee2e6'})
)


🧹 Cleaning and aligning datasets...
✅ Data cleaning complete. Previewing cleaned sentiment data:


C:\Users\mahes\AppData\Local\Temp\ipykernel_102532\1021104134.py:14: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  df['date'] = df['datetime_utc'].dt.floor('d')


,date,sentiment_score,sentiment_class
0,2018-02-01 00:00:00,30,Fear
1,2018-02-02 00:00:00,15,Extreme Fear
2,2018-02-03 00:00:00,40,Fear


In [13]:
def engineer_sentiment_features(df_sentiment: pd.DataFrame) -> pd.DataFrame:
    df = df_sentiment.copy().sort_values('date')
    
    # Create temporal lag features (Crucial for predictive modeling later)
    df['sentiment_class_lag1'] = df['sentiment_class'].shift(1)
    df['sentiment_score_lag1'] = df['sentiment_score'].shift(1)
    
    # Identify Market Regime Transitions (e.g. Fear -> Greed)
    df['sentiment_transition'] = df.apply(
        lambda x: f"{x['sentiment_class_lag1']} -> {x['sentiment_class']}" 
        if pd.notna(x['sentiment_class_lag1']) else 'Unknown', axis=1
    )
    
    # Boolean flag for days where the market changes its mind
    df['is_transition_day'] = (df['sentiment_class'] != df['sentiment_class_lag1']) & pd.notna(df['sentiment_class_lag1'])
    return df

print("⚙️ Engineering Temporal Sentiment Features...")
df_sentiment_feat = engineer_sentiment_features(df_sentiment_clean)
print("✅ Feature engineering complete. Previewing transition states:")

# Display the final features with premium formatting
display(
    df_sentiment_feat[['date', 'sentiment_class', 'sentiment_transition']].tail(3).style
    .set_caption("Engineered Features: Market Sentiment Transitions")
    .set_table_styles([{
        'selector': 'caption',
        'props': [('color', '#2c3e50'), ('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]
    }, {
        'selector': 'th',
        'props': [('background-color', '#34495e'), ('color', 'white'), ('font-weight', 'bold')]
    }])
    .set_properties(**{'background-color': '#f8f9fa', 'color': '#212529', 'border': '1px solid #dee2e6'})
)


⚙️ Engineering Temporal Sentiment Features...
✅ Feature engineering complete. Previewing transition states:


,date,sentiment_class,sentiment_transition
2641,2025-04-30 00:00:00,Greed,Greed -> Greed
2642,2025-05-01 00:00:00,Neutral,Greed -> Neutral
2643,2025-05-02 00:00:00,Greed,Neutral -> Greed


In [16]:
def aggregate_trader_features(df_trader: pd.DataFrame) -> pd.DataFrame:
    grouped = df_trader.groupby(['date', 'Account'])
    
    daily_pnl = grouped['Closed PnL'].sum()
    num_trades = grouped.size()
    winning_trades = df_trader[df_trader['Closed PnL'] > 0].groupby(['date', 'Account']).size()
    losing_trades = df_trader[df_trader['Closed PnL'] < 0].groupby(['date', 'Account']).size()
    
    df_agg = pd.DataFrame({
        'daily_pnl': daily_pnl, 'num_trades': num_trades,
        'winning_trades': winning_trades, 'losing_trades': losing_trades
    }).fillna(0)
    
    df_agg['win_rate'] = (df_agg['winning_trades'] / (df_agg['winning_trades'] + df_agg['losing_trades'])).fillna(0)
    df_agg['avg_trade_size_usd'] = grouped['Size USD'].mean()
    df_agg['total_volume_usd'] = grouped['Size USD'].sum()
    
    longs = df_trader[df_trader['Side'] == 'BUY'].groupby(['date', 'Account']).size()
    shorts = df_trader[df_trader['Side'] == 'SELL'].groupby(['date', 'Account']).size()

    df_agg['long_trades'] = longs
    df_agg['short_trades'] = shorts
    df_agg.fillna({'long_trades': 0, 'short_trades': 0}, inplace=True)
    
    # 🚨 FIX: Replaced corrupted long_short_ratio with a bounded percentage (0 to 1)
    total_direction_trades = df_agg['long_trades'] + df_agg['short_trades']
    df_agg['long_pct'] = df_agg['long_trades'] / total_direction_trades.replace(0, 1)
    
    return df_agg.reset_index()

print("📈 Aggregating Trader behavior metrics...")
df_trader_agg = aggregate_trader_features(df_trader_clean)
print("✅ Aggregation complete. Previewing trader metrics:")

# Display the fixed features with premium formatting
display(
    df_trader_agg.head(3).style
    .set_caption("Aggregated Features: Daily Trader Profiles")
    .format({
        'daily_pnl': '${:,.2f}',
        'avg_trade_size_usd': '${:,.2f}',
        'total_volume_usd': '${:,.2f}',
        'win_rate': '{:.1%}',
        'long_pct': '{:.1%}'
    })
    .set_table_styles([{
        'selector': 'caption',
        'props': [('color', '#2c3e50'), ('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]
    }, {
        'selector': 'th',
        'props': [('background-color', '#34495e'), ('color', 'white'), ('font-weight', 'bold')]
    }])
    .set_properties(**{'background-color': '#f8f9fa', 'color': '#212529', 'border': '1px solid #dee2e6'})
)


📈 Aggregating Trader behavior metrics...
✅ Aggregation complete. Previewing trader metrics:


,date,Account,daily_pnl,num_trades,winning_trades,losing_trades,win_rate,avg_trade_size_usd,total_volume_usd,long_trades,short_trades,long_pct
0,2023-03-28 00:00:00,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,$0.00,3,0.000000,0.000000,0.0%,$159.00,$477.00,3.000000,0.000000,100.0%
1,2023-11-14 00:00:00,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,$0.00,2,0.000000,0.000000,0.0%,"$23,066.94","$46,133.87",2.000000,0.000000,100.0%
2,2023-11-14 00:00:00,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,$155.50,1043,287.000000,196.000000,59.4%,"$11,034.80","$11,509,295.89",489.000000,554.000000,46.9%


In [17]:
print("🔗 Merging trader and sentiment datasets...")
df_merged = pd.merge(df_trader_agg, df_sentiment_feat, on='date', how='inner')

# 🚨 FIX: Transparent Dataset Size Reporting
print("-" * 50)
print(f"📊 Merged Dataset Overview:")
print(f"   ➤ Overlapping Date Range: {df_merged['date'].min().strftime('%Y-%m-%d')} to {df_merged['date'].max().strftime('%Y-%m-%d')}")
print(f"   ➤ Total Merged Rows:      {len(df_merged):,} rows")
print(f"   ➤ Unique Traders:         {df_merged['Account'].nunique():,} accounts")
print("-" * 50)

print("🔄 Calculating rolling PnL and lagged predictive features...")
df_merged = df_merged.sort_values(['Account', 'date'])

# Calculate 3-day rolling PnL
df_merged['rolling_3d_pnl'] = df_merged.groupby('Account')['daily_pnl'].transform(lambda x: x.rolling(3, min_periods=1).sum())

# 🚨 FIX: Create lagged behavioral features (Crucial for preventing Data Leakage in Phase 3)
# We MUST use yesterday's behavior to predict today's profitability!
lag_features = ['num_trades', 'win_rate', 'avg_trade_size_usd', 'long_pct', 'total_volume_usd']
for feat in lag_features:
    df_merged[f'{feat}_lag1'] = df_merged.groupby('Account')[feat].shift(1)

output_path = '../output/merged_trader_sentiment.csv'
df_merged.to_csv(output_path, index=False)

print(f"✅ Phase 1 Complete! Clean dataset saved to: {output_path}")

# Display the final Phase 1 output with premium formatting
display(
    df_merged[['date', 'Account', 'daily_pnl', 'long_pct', 'sentiment_class', 'win_rate_lag1']].head(3).style
    .set_caption("Final Merged Data (Showing New Lagged Features)")
    .format({
        'daily_pnl': '${:,.2f}',
        'long_pct': '{:.1%}',
        'win_rate_lag1': '{:.1%}'
    })
    .set_table_styles([{
        'selector': 'caption',
        'props': [('color', '#2c3e50'), ('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]
    }, {
        'selector': 'th',
        'props': [('background-color', '#34495e'), ('color', 'white'), ('font-weight', 'bold')]
    }])
    .set_properties(**{'background-color': '#f8f9fa', 'color': '#212529', 'border': '1px solid #dee2e6'})
)


🔗 Merging trader and sentiment datasets...
--------------------------------------------------
📊 Merged Dataset Overview:
   ➤ Overlapping Date Range: 2023-03-28 to 2025-02-19
   ➤ Total Merged Rows:      77 rows
   ➤ Unique Traders:         32 accounts
--------------------------------------------------
🔄 Calculating rolling PnL and lagged predictive features...
✅ Phase 1 Complete! Clean dataset saved to: ../output/merged_trader_sentiment.csv


,date,Account,daily_pnl,long_pct,sentiment_class,win_rate_lag1
16,2024-10-27 00:00:00,0x083384f897ee0f19899168e3b1bec365f52a9012,"$-327,505.90",30.1%,Greed,nan%
45,2025-02-19 00:00:00,0x083384f897ee0f19899168e3b1bec365f52a9012,"$1,927,735.72",46.8%,Fear,8.6%
17,2024-10-27 00:00:00,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,"$20,607.45",45.3%,Greed,nan%
